# LeetCode 986 — Interval List Intersections bằng Python

Link đề: https://leetcode.com/problems/interval-list-intersections/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers

Skeleton LeetCode:

```python
class Solution:
    def intervalIntersection(self, firstList: List[List[int]], secondList: List[List[int]]) -> List[List[int]]:
        ...
```

Ghi chú: đáp án **duy nhất** (danh sách giao đã sắp xếp theo thứ tự tăng, các đoạn rời nhau). Bài dùng **đoạn đóng** `[a, b]` nên giao có thể chỉ là một **điểm** (`[5, 5]`) — bẫy chính: dùng `<` thay `<=` sẽ làm mất các giao điểm. Oracle độc lập: biểu diễn mỗi đoạn bằng **tập điểm trên lưới nửa số nguyên** rồi lấy giao tập hợp.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: hai danh sách đoạn đóng, mỗi danh sách gồm các đoạn **rời nhau và đã sắp xếp**; trả về danh sách mọi giao (không rỗng) giữa một đoạn của danh sách này và một đoạn của danh sách kia.
2. Nắm công cụ: hai con trỏ, `bisect`, quét sự kiện (sweep line), giao hai đoạn `[max(a₁, b₁), min(a₂, b₂)]`.
3. Đi từ **thử mọi cặp (`O(nm)`)** → **hai con trỏ** → **quét sự kiện** → **chặt nhị phân** → **trộn theo điểm đầu và nhớ đoạn cuối của danh sách kia**.
4. Hiểu mấu chốt: giao của `[a₁, a₂]` và `[b₁, b₂]` là `[lo, hi] = [max(a₁, b₁), min(a₂, b₂)]`, **không rỗng ⇔ `lo ≤ hi`**; sau khi xét một cặp, đoạn có **điểm cuối nhỏ hơn** không thể giao với đoạn nào sau nữa của danh sách kia ⇒ tiến con trỏ đó.
5. Tránh lỗi kinh điển: `lo < hi` (mất giao điểm `[5,5]`), tiến sai con trỏ (theo điểm đầu thay vì điểm cuối), tiến cả hai, quên xử lý danh sách rỗng, không kiểm tra `lo ≤ hi`, thứ tự sự kiện sai khi quét (kết thúc trước bắt đầu tại cùng toạ độ).

## 2. Đề bài & ví dụ

Cho hai danh sách đoạn **đóng** `firstList` và `secondList`, `firstList[i] = [start_i, end_i]`, `secondList[j] = [start_j, end_j]`. Mỗi danh sách gồm các đoạn **rời nhau** và **đã sắp xếp**. Trả về **giao** của hai danh sách.

Đoạn đóng `[a, b]` (`a ≤ b`) là tập số thực `x` với `a ≤ x ≤ b`. Giao của hai đoạn đóng là tập rỗng hoặc một đoạn đóng; ví dụ giao của `[1,3]` và `[2,4]` là `[2,3]`.

**Ví dụ 1:** `firstList = [[0,2],[5,10],[13,23],[24,25]]`, `secondList = [[1,5],[8,12],[15,24],[25,26]]` → `[[1,2],[5,5],[8,10],[15,23],[24,24],[25,25]]`.

**Ví dụ 2:** `firstList = [[1,3],[5,9]]`, `secondList = []` → `[]`.

**Ràng buộc:** `0 <= firstList.length, secondList.length <= 1000`; tổng độ dài `≥ 1`; `0 <= start < end <= 10^9`; trong mỗi danh sách `end_i < start_{i+1}`.

In [1]:
import bisect
import heapq
import itertools
import random
import time
from typing import List

ex1 = ([[0, 2], [5, 10], [13, 23], [24, 25]], [[1, 5], [8, 12], [15, 24], [25, 26]])
examples = [(ex1, [[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]), (([[1, 3], [5, 9]], []), [])]
for (a, b), want in examples:
    print(f"first = {a}, second = {b} -> {want}")

first = [[0, 2], [5, 10], [13, 23], [24, 25]], second = [[1, 5], [8, 12], [15, 24], [25, 26]] -> [[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]
first = [[1, 3], [5, 9]], second = [] -> []


## 3. Kiến thức nền

### 3.1 Giao của hai đoạn đóng

`[a₁, a₂] ∩ [b₁, b₂] = [max(a₁, b₁), min(a₂, b₂)]` nếu `max(a₁, b₁) ≤ min(a₂, b₂)`, ngược lại rỗng. Dấu `≤` quan trọng vì đoạn **đóng**: `[1, 5] ∩ [5, 10] = [5, 5]` (một điểm).

In [2]:
def intersect(x, y):
    lo, hi = max(x[0], y[0]), min(x[1], y[1])
    return [lo, hi] if lo <= hi else None


print(intersect([1, 3], [2, 4]), intersect([1, 5], [5, 10]), intersect([1, 2], [3, 4]), intersect([0, 10], [3, 4]))

[2, 3] [5, 5] None [3, 4]


### 3.2 Hai danh sách đã sắp xếp và rời nhau

Trong mỗi danh sách mọi `end_i < start_{i+1}`, nên các đoạn tăng dần theo cả điểm đầu và điểm cuối. Tính chất này cho phép duyệt **một lượt** bằng hai con trỏ (giống bước "trộn" trong merge sort). Ngoài ra, các giao tìm được cũng **rời nhau** và được sinh ra theo thứ tự tăng.

### 3.3 Oracle bằng tập điểm trên lưới nửa số nguyên

Để có một oracle không dùng công thức `max/min`, ta biểu diễn đoạn `[a, b]` bằng tập các số nguyên `2a, 2a+1, …, 2b` (toạ độ nhân đôi, tức bước `0.5`). Giao hai đoạn = giao hai tập; một **dải liên tiếp tối đa** các điểm nguyên là một đoạn của đáp án. Bước `0.5` đảm bảo hai đoạn `[1,2]` và `[3,4]` vẫn **tách rời** (còn điểm `2.5` ở giữa) nhưng `[1,2]` và `[2,3]` dính liền ở điểm `2`.

In [3]:
def to_points(intervals):
    pts = set()
    for a, b in intervals:
        pts.update(range(2 * a, 2 * b + 1))
    return pts


def runs_to_intervals(points):
    """Các dải liên tiếp tối đa của tập điểm (toạ độ nhân đôi) -> danh sách đoạn [lo, hi] (toạ độ gốc)."""
    out, ps = [], sorted(points)
    i = 0
    while i < len(ps):
        j = i
        while j + 1 < len(ps) and ps[j + 1] == ps[j] + 1:
            j += 1
        assert ps[i] % 2 == 0 and ps[j] % 2 == 0, "đầu mút không nguyên"
        out.append([ps[i] // 2, ps[j] // 2])
        i = j + 1
    return out


def oracle_lattice(first, second):
    return runs_to_intervals(to_points(first) & to_points(second))


for (a, b), want in examples:
    assert oracle_lattice(a, b) == want
print(oracle_lattice(*ex1))

[[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]


## 4. Ý tưởng nền tảng / trực giác

Trên trục số, vẽ hai danh sách. Ví dụ 1: ở `first` có `[0,2]`, `[5,10]`, `[13,23]`, `[24,25]`; ở `second` có `[1,5]`, `[8,12]`, `[15,24]`, `[25,26]`. Hai đoạn giao nhau khi chồng lên nhau; vì cả hai danh sách tăng dần, ta chỉ cần "đi" dọc trục số: so hai đoạn hiện tại, ghi giao (nếu có), rồi **bỏ đoạn kết thúc sớm hơn** (nó không thể giao với đoạn nào xa hơn của danh sách kia vì chúng bắt đầu muộn hơn điểm cuối của nó).

`[1,5]` và `[5,10]` chung đúng điểm `5` (đoạn đóng) ⇒ `[5,5]`; tương tự `[15,24]` và `[24,25]` ⇒ `[24,24]`, `[24,25]` và `[25,26]` ⇒ `[25,25]`.

## 5. Brute force — thử mọi cặp đoạn

**Ý tưởng:** với mọi `i`, `j` tính `lo = max(start)`, `hi = min(end)`; nếu `lo ≤ hi` thì ghi `[lo, hi]`. Vì duyệt `i` tăng rồi `j` tăng, kết quả phải **sắp xếp** theo `lo` (ta sắp xếp cuối cho chắc, hoặc chứng minh không cần — mục 10). `O(nm)` thời gian.

In [4]:
def intersect_brute(first, second):
    res = []
    for a in first:
        for b in second:
            lo, hi = max(a[0], b[0]), min(a[1], b[1])
            if lo <= hi:
                res.append([lo, hi])
    res.sort()
    return res


for (a, b), want in examples:
    assert intersect_brute(a, b) == want == oracle_lattice(a, b)
print(intersect_brute(*ex1))

[[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]


In [5]:
# Code nộp LeetCode cho Brute force (O(n·m) = 10^6: vẫn qua với n, m <= 1000)
class SolutionBrute:
    def intervalIntersection(self, firstList: List[List[int]], secondList: List[List[int]]) -> List[List[int]]:
        res = []
        for a in firstList:
            for b in secondList:
                lo, hi = max(a[0], b[0]), min(a[1], b[1])
                if lo <= hi:
                    res.append([lo, hi])
        res.sort()
        return res

**Độ phức tạp:** `O(n·m)` thời gian (`≤ 10^6` bước với `n, m ≤ 1000`), `O(k)` bộ nhớ cho `k` giao.

**Nút thắt:** ta so mọi cặp trong khi hai danh sách đã sắp xếp: với mỗi đoạn chỉ có vài đoạn "lân cận" của danh sách kia giao với nó.

## 6. Bước đột phá

Hai con trỏ `i`, `j`. So `first[i]` với `second[j]`, ghi giao nếu có. Sau đó tiến con trỏ của đoạn có **điểm cuối nhỏ hơn** (đoạn này không thể giao với `second[j+1]` hay `first[i+1]` kế tiếp... cụ thể, nó không thể giao với các đoạn sau của danh sách kia vì chúng bắt đầu sau điểm cuối của đoạn còn lại). Mỗi bước tiến một con trỏ ⇒ `O(n + m)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** hai con trỏ (đáp án chính), **Cách 3** quét sự kiện, **Cách 4** chặt nhị phân, **Cách 5** trộn theo điểm đầu và nhớ đoạn cuối của danh sách kia.

### Cách 2 — Hai con trỏ

**Giải thích từng đoạn:**

1. `i = j = 0`.
2. Khi cả hai còn đoạn: `lo = max(a.start, b.start)`, `hi = min(a.end, b.end)`; nếu `lo <= hi` thì ghi `[lo, hi]`.
3. Tiến con trỏ của đoạn có điểm cuối **nhỏ hơn** (nếu bằng nhau tiến một trong hai là đủ — đoạn kia không giao thêm với đoạn nào, vì đoạn kế tiếp của danh sách vừa tiến bắt đầu sau điểm cuối chung).

In [6]:
def intersect_two_pointers(first, second):
    i = j = 0
    res = []
    while i < len(first) and j < len(second):
        lo = max(first[i][0], second[j][0])
        hi = min(first[i][1], second[j][1])
        if lo <= hi:                                    # <= vì đoạn đóng: giao có thể là một điểm
            res.append([lo, hi])
        if first[i][1] < second[j][1]:                  # bỏ đoạn kết thúc sớm hơn
            i += 1
        else:
            j += 1
    return res


for (a, b), want in examples:
    assert intersect_two_pointers(a, b) == want
print(intersect_two_pointers(*ex1))

[[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]


In [7]:
# Code nộp LeetCode cho Cách 2
class SolutionTwoPointers:
    def intervalIntersection(self, firstList: List[List[int]], secondList: List[List[int]]) -> List[List[int]]:
        i = j = 0
        res = []
        while i < len(firstList) and j < len(secondList):
            lo = max(firstList[i][0], secondList[j][0])
            hi = min(firstList[i][1], secondList[j][1])
            if lo <= hi:
                res.append([lo, hi])
            if firstList[i][1] < secondList[j][1]:
                i += 1
            else:
                j += 1
        return res

**Độ phức tạp:** `O(n + m)` thời gian, `O(1)` bộ nhớ phụ (ngoài kết quả).

### Cách 3 — Quét sự kiện (sweep line)

**Ý tưởng:** biến mỗi đoạn thành hai sự kiện: bắt đầu tại `s`, kết thúc tại `e`. Sắp xếp theo `(toạ độ, loại)` với **bắt đầu trước kết thúc** (cùng toạ độ) — để đoạn `[1,5]` và `[5,10]` vẫn "chồng" tại điểm `5`. Duyệt, giữ **độ sâu** (số đoạn đang mở, tối đa `2` vì mỗi danh sách rời nhau): khi độ sâu tăng lên `2` tại toạ độ `x` thì mở một giao bắt đầu tại `x`; khi gặp sự kiện kết thúc lúc độ sâu đang `2` thì đóng giao tại toạ độ đó.

In [8]:
def intersect_sweep(first, second):
    events = []
    for s, e in first + second:
        events.append((s, 0))                           # 0: bắt đầu (xử lý trước)
        events.append((e, 1))                           # 1: kết thúc
    events.sort()
    depth, open_at, res = 0, None, []
    for x, kind in events:
        if kind == 0:
            depth += 1
            if depth == 2:
                open_at = x
        else:
            if depth == 2:
                res.append([open_at, x])
            depth -= 1
    return res


for (a, b), want in examples:
    assert intersect_sweep(a, b) == want
print(intersect_sweep(*ex1))

[[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]


In [9]:
# Code nộp LeetCode cho Cách 3
class SolutionSweep:
    def intervalIntersection(self, firstList: List[List[int]], secondList: List[List[int]]) -> List[List[int]]:
        events = []
        for s, e in firstList + secondList:
            events.append((s, 0))
            events.append((e, 1))
        events.sort()
        depth, open_at, res = 0, 0, []
        for x, kind in events:
            if kind == 0:
                depth += 1
                if depth == 2:
                    open_at = x
            else:
                if depth == 2:
                    res.append([open_at, x])
                depth -= 1
        return res

**Độ phức tạp:** `O((n + m) log(n + m))` do sắp xếp; `O(n + m)` bộ nhớ.

### Cách 4 — Chặt nhị phân

**Ý tưởng:** với mỗi đoạn `a` của `first`, tìm bằng `bisect` đoạn đầu tiên `b` của `second` có `b.end ≥ a.start`, rồi duyệt tiếp các `b` có `b.start ≤ a.end`, ghi giao. Mảng `ends = [b.end]` đã tăng dần nên `bisect_left(ends, a.start)` cho chỉ số bắt đầu.

In [10]:
def intersect_bisect(first, second):
    ends = [b[1] for b in second]
    res = []
    for a in first:
        j = bisect.bisect_left(ends, a[0])             # đoạn đầu tiên của second có end >= a.start
        while j < len(second) and second[j][0] <= a[1]:
            lo, hi = max(a[0], second[j][0]), min(a[1], second[j][1])
            res.append([lo, hi])                       # đã đảm bảo lo <= hi
            j += 1
    return res


for (a, b), want in examples:
    assert intersect_bisect(a, b) == want
print(intersect_bisect(*ex1))

[[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]


In [11]:
# Code nộp LeetCode cho Cách 4
import bisect


class SolutionBisect:
    def intervalIntersection(self, firstList: List[List[int]], secondList: List[List[int]]) -> List[List[int]]:
        ends = [b[1] for b in secondList]
        res = []
        for a in firstList:
            j = bisect.bisect_left(ends, a[0])
            while j < len(secondList) and secondList[j][0] <= a[1]:
                res.append([max(a[0], secondList[j][0]), min(a[1], secondList[j][1])])
                j += 1
        return res

**Độ phức tạp:** `O(n log m + k)` với `k` là số giao (mỗi `j` duyệt đều sinh một giao; tổng `k ≤ n + m − 1`).

### Cách 5 — Trộn theo điểm đầu và nhớ đoạn cuối của danh sách kia

**Ý tưởng:** gộp hai danh sách theo `(start, nguồn)`. Duyệt theo thứ tự đó, giữ `last[0]`, `last[1]` là đoạn gần nhất đã gặp của mỗi danh sách. Khi gặp đoạn `x` của danh sách `s`, chỉ cần so với `last[1 − s]`: giao `[x.start, min(x.end, last.end)]` nếu `last.end ≥ x.start`. Vì danh sách kia rời nhau và các đoạn trước đó bắt đầu sớm hơn `last`, chỉ `last` mới có thể chồng lên `x` (xem mục 10).

In [12]:
def intersect_merge_by_start(first, second):
    merged = heapq.merge(((a[0], 0, a[1]) for a in first), ((b[0], 1, b[1]) for b in second))
    last = [None, None]                                  # đoạn (end) gần nhất của mỗi danh sách
    res = []
    for start, src, end in merged:
        other = last[1 - src]
        if other is not None and other >= start:         # đoạn kia (bắt đầu <= start) còn phủ tới start
            res.append([start, min(end, other)])
        last[src] = end
    return res


for (a, b), want in examples:
    assert intersect_merge_by_start(a, b) == want
print(intersect_merge_by_start(*ex1))

[[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]


In [13]:
# Code nộp LeetCode cho Cách 5
import heapq


class SolutionMergeByStart:
    def intervalIntersection(self, firstList: List[List[int]], secondList: List[List[int]]) -> List[List[int]]:
        merged = heapq.merge(((a[0], 0, a[1]) for a in firstList), ((b[0], 1, b[1]) for b in secondList))
        last = [None, None]
        res = []
        for start, src, end in merged:
            other = last[1 - src]
            if other is not None and other >= start:
                res.append([start, min(end, other)])
            last[src] = end
        return res

**Độ phức tạp:** `O(n + m)` thời gian (hai danh sách đã sắp ⇒ `heapq.merge` tuyến tính theo số phần tử), `O(1)` bộ nhớ phụ.

## 8. TRACE từng bước

**Trace Cách 2** với ví dụ 1:

| `i` | `j` | `first[i]` | `second[j]` | `lo` | `hi` | ghi | tiến |
|---|---|---|---|---|---|---|---|
| 0 | 0 | `[0,2]` | `[1,5]` | 1 | 2 | `[1,2]` | `i` (2 < 5) |
| 1 | 0 | `[5,10]` | `[1,5]` | 5 | 5 | `[5,5]` | `j` (10 ≥ 5) |
| 1 | 1 | `[5,10]` | `[8,12]` | 8 | 10 | `[8,10]` | `i` (10 < 12) |
| 2 | 1 | `[13,23]` | `[8,12]` | 13 | 12 | — | `j` |
| 2 | 2 | `[13,23]` | `[15,24]` | 15 | 23 | `[15,23]` | `i` |
| 3 | 2 | `[24,25]` | `[15,24]` | 24 | 24 | `[24,24]` | `j` |
| 3 | 3 | `[24,25]` | `[25,26]` | 25 | 25 | `[25,25]` | `i` ⇒ hết |

Hàm trace in từng bước:

In [14]:
def trace_two_pointers(first, second):
    i = j = 0
    res = []
    while i < len(first) and j < len(second):
        a, b = first[i], second[j]
        lo, hi = max(a[0], b[0]), min(a[1], b[1])
        got = lo <= hi
        if got:
            res.append([lo, hi])
        move = "i" if a[1] < b[1] else "j"
        print(f"  i={i} j={j}: {a} ∩ {b}: lo={lo}, hi={hi} -> {'ghi ' + str([lo, hi]) if got else 'rỗng'} | tiến {move}")
        if a[1] < b[1]:
            i += 1
        else:
            j += 1
    print("=>", res)
    return res


assert trace_two_pointers(*ex1) == examples[0][1]

  i=0 j=0: [0, 2] ∩ [1, 5]: lo=1, hi=2 -> ghi [1, 2] | tiến i
  i=1 j=0: [5, 10] ∩ [1, 5]: lo=5, hi=5 -> ghi [5, 5] | tiến j
  i=1 j=1: [5, 10] ∩ [8, 12]: lo=8, hi=10 -> ghi [8, 10] | tiến i
  i=2 j=1: [13, 23] ∩ [8, 12]: lo=13, hi=12 -> rỗng | tiến j
  i=2 j=2: [13, 23] ∩ [15, 24]: lo=15, hi=23 -> ghi [15, 23] | tiến i
  i=3 j=2: [24, 25] ∩ [15, 24]: lo=24, hi=24 -> ghi [24, 24] | tiến j
  i=3 j=3: [24, 25] ∩ [25, 26]: lo=25, hi=25 -> ghi [25, 25] | tiến i
=> [[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]


**Trace Cách 3 (quét sự kiện)** với `first = [[1,5]]`, `second = [[5,9]]`: sự kiện sắp xếp `(1,bắt đầu) (5,bắt đầu) (5,kết thúc) (9,kết thúc)`; bắt đầu trước kết thúc tại toạ độ `5` ⇒ độ sâu lên `2` tại `5`, rồi kết thúc đầu tiên tại `5` đóng giao `[5,5]`. Nếu để kết thúc đi trước bắt đầu, độ sâu không bao giờ đạt `2` ⇒ mất giao điểm (xem mục 13).

In [15]:
def trace_sweep(first, second):
    events = sorted([(s, 0) for s, e in first + second] + [(e, 1) for s, e in first + second])
    depth, open_at, res = 0, None, []
    for x, kind in events:
        if kind == 0:
            depth += 1
            if depth == 2:
                open_at = x
            print(f"  x={x}: bắt đầu -> độ sâu {depth}")
        else:
            if depth == 2:
                res.append([open_at, x])
            depth -= 1
            print(f"  x={x}: kết thúc -> độ sâu {depth}" + (f" | đóng giao {res[-1]}" if res and res[-1][1] == x and depth == 1 else ""))
    print("=>", res)
    return res


assert trace_sweep([[1, 5]], [[5, 9]]) == [[5, 5]]

  x=1: bắt đầu -> độ sâu 1
  x=5: bắt đầu -> độ sâu 2
  x=5: kết thúc -> độ sâu 1 | đóng giao [5, 5]
  x=9: kết thúc -> độ sâu 0
=> [[5, 5]]


## 9. Code nộp LeetCode

Chọn **Cách 2** (hai con trỏ): `O(n + m)`, ngắn, đúng. Giữ nguyên `class Solution`, `intervalIntersection(self, firstList, secondList)`:

In [16]:
class Solution:
    def intervalIntersection(self, firstList: List[List[int]], secondList: List[List[int]]) -> List[List[int]]:
        i = j = 0
        res = []
        while i < len(firstList) and j < len(secondList):
            lo = max(firstList[i][0], secondList[j][0])    # đầu giao = max hai đầu
            hi = min(firstList[i][1], secondList[j][1])    # cuối giao = min hai cuối
            if lo <= hi:                                   # đoạn đóng: lo == hi vẫn là giao (một điểm)
                res.append([lo, hi])
            if firstList[i][1] < secondList[j][1]:         # bỏ đoạn kết thúc sớm hơn
                i += 1
            else:
                j += 1
        return res


print(Solution().intervalIntersection(*ex1))
assert all(Solution().intervalIntersection(*args) == want for args, want in examples)

[[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]


## 10. Vì sao đúng

**(a) Công thức giao.** `[a₁,a₂] ∩ [b₁,b₂] = {x : a₁ ≤ x ≤ a₂ và b₁ ≤ x ≤ b₂} = {x : max(a₁,b₁) ≤ x ≤ min(a₂,b₂)}`. Khác rỗng ⇔ `lo ≤ hi`. Đây là đoạn đóng; `lo = hi` cho một điểm.

**(b) Hai con trỏ không bỏ sót.** Giả sử cặp `(F_p, S_q)` giao nhau; ta chứng minh nó được xét. Bất biến: mọi cặp `(F_{p'}, S_{q'})` với `p' < i` hoặc `q' < j` **không giao nhau** (hoặc đã được ghi). Bước tiến: nếu `F_i.end < S_j.end`, ta bỏ `F_i`. Cặp `(F_i, S_{j'})` với `j' > j` có `S_{j'}.start > S_j.end > F_i.end`, nên không giao ⇒ `F_i` không còn cặp giao nào chưa xét. Trường hợp đối xứng tiến `j`. Nếu `F_i.end = S_j.end` ta tiến `j` (nhánh `else`): cặp `(F_{i'}, S_j)` với `i' > i` có `F_{i'}.start > F_i.end = S_j.end` ⇒ không giao; còn `(F_i, S_{j'})` với `j' > j` cũng có `S_{j'}.start > S_j.end = F_i.end` ⇒ không giao. Vậy bỏ `S_j` an toàn.

**(c) Thứ tự và tính rời nhau.** Các giao ghi ra có `lo` không giảm (con trỏ chỉ tiến). Vì `hi ≤` điểm cuối của một đoạn trong mỗi danh sách và đoạn kế tiếp bắt đầu sau điểm cuối đó, giao kế tiếp có `lo` lớn hơn nghiêm ngặt `hi` hiện tại. Vậy đầu ra đã sắp xếp, rời nhau và không cần gộp.

**(d) Cách 1** liệt kê đủ mọi cặp rồi sắp xếp: hiển nhiên đúng.

**(e) Cách 3.** Độ sâu = số đoạn chứa điểm hiện tại. Mỗi danh sách rời nhau nên độ sâu `≤ 2`. Độ sâu `= 2` ⇔ điểm thuộc cả hai danh sách. Khi bắt đầu xử lý trước kết thúc tại cùng toạ độ, một điểm chung `x` (hai đoạn chạm tại `x`) được tính là độ sâu `2`. Một lần độ sâu đạt `2` tại `x` rồi kết thúc đầu tiên tại `y` cho đúng một giao `[x, y]` (mọi điểm giữa nằm trong cả hai đoạn).

**(f) Cách 4.** `ends` tăng dần. `bisect_left(ends, a.start)` cho đoạn đầu tiên của `second` có `end ≥ a.start`; các đoạn trước đó kết thúc trước `a.start` nên không giao. Từ đó đến khi `start > a.end`, mọi đoạn đều giao (vì `end ≥ a.start` do tăng dần và `start ≤ a.end`).

**(g) Cách 5.** Duyệt theo `(start, nguồn)`. Với đoạn `x` (danh sách `s`) và đoạn `y` của danh sách kia: nếu `y.start ≤ x.start` thì `y` được xử lý trước (hoặc cùng toạ độ, nguồn nhỏ hơn đi trước) — khi xử lý `x`, `last[1−s]` là đoạn **muộn nhất** của danh sách kia có `start ≤ x.start`; mọi đoạn trước nó kết thúc trước `last.start ≤ x.start` nên không chồng `x`. Nếu `y.start > x.start` thì cặp được phát hiện khi xử lý `y` (với `x` hoặc đoạn muộn hơn của `s`, đúng như lập luận đối xứng). Mỗi cặp giao nhau được sinh đúng một lần (ở đoạn có `(start, nguồn)` lớn hơn) và `[start, min(end, other)]` là giao (`start` là điểm đầu muộn hơn).

**(h) Kiểm chứng bằng máy.** Mọi cách khớp oracle tập điểm trên lưới nửa số nguyên (không dùng `max/min`) trên **mọi** cặp danh sách trong `[0, 8]` (`256² = 65 536` cặp) và khớp nhau tới `n, m = 1000` với toạ độ tới `10^9` (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi cặp | `O(n·m)` | `O(k)` | oracle thứ hai, qua được với `n, m ≤ 1000` |
| 2. Hai con trỏ | `O(n + m)` | `O(1)` | **đáp án chính** |
| 3. Quét sự kiện | `O((n+m) log(n+m))` | `O(n+m)` | tổng quát hoá cho `k` danh sách |
| 4. Chặt nhị phân | `O(n log m + k)` | `O(m)` | khi `n ≪ m` |
| 5. Trộn theo điểm đầu | `O(n + m)` | `O(1)` | dạng "nhớ đoạn cuối" |

(`k` là số giao, `k ≤ n + m − 1`.) Đo thực tế `n = m = 1000`:

In [17]:
def make_list(rng, n, hi=10 ** 9):
    """n đoạn rời nhau, sắp xếp, 0 <= start < end <= hi, end_i < start_{i+1}."""
    pts = sorted(rng.sample(range(hi + 1), 2 * n))
    return [[pts[2 * i], pts[2 * i + 1]] for i in range(n)]


def bench(fn, args, repeat=5):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(*args)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(986)
fa, sb = make_list(rng, 1000), make_list(rng, 1000)
for name, fn in [("two_pointers", intersect_two_pointers), ("sweep", intersect_sweep), ("bisect", intersect_bisect), ("merge_by_start", intersect_merge_by_start), ("brute", intersect_brute)]:
    print(f"{name:15s}: {bench(fn, (fa, sb), repeat=2 if name == 'brute' else 20) * 1e3:7.2f} ms (n = m = 1000)")

two_pointers   :    0.54 ms (n = m = 1000)
sweep          :    0.62 ms (n = m = 1000)


bisect         :    0.43 ms (n = m = 1000)


merge_by_start :    0.54 ms (n = m = 1000)


brute          :   84.70 ms (n = m = 1000)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi cặp danh sách nhỏ với oracle tập điểm, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [18]:
IMPLS = {
    "two_pointers": intersect_two_pointers,
    "sweep": intersect_sweep,
    "bisect": intersect_bisect,
    "merge_by_start": intersect_merge_by_start,
    "brute": intersect_brute,
    "Solution": lambda a, b: Solution().intervalIntersection([x[:] for x in a], [x[:] for x in b]),
    "SolutionTwoPointers": lambda a, b: SolutionTwoPointers().intervalIntersection(a, b),
    "SolutionSweep": lambda a, b: SolutionSweep().intervalIntersection(a, b),
    "SolutionBisect": lambda a, b: SolutionBisect().intervalIntersection(a, b),
    "SolutionMergeByStart": lambda a, b: SolutionMergeByStart().intervalIntersection(a, b),
    "SolutionBrute": lambda a, b: SolutionBrute().intervalIntersection(a, b),
}


def oracle(first, second):
    """Oracle tập điểm với NÉN TOẠ ĐỘ (dùng được cho toạ độ tới 10^9): mỗi toạ độ phân biệt có chỉ số i,
    điểm 2i là chính toạ độ đó, điểm 2i+1 là khoảng mở giữa toạ độ i và i+1."""
    coords = sorted({v for s, e in first + second for v in (s, e)})
    idx = {v: i for i, v in enumerate(coords)}

    def pts(lst):
        out = set()
        for a, b in lst:
            out.update(range(2 * idx[a], 2 * idx[b] + 1))
        return out

    ps = sorted(pts(first) & pts(second))
    res, i = [], 0
    while i < len(ps):
        j = i
        while j + 1 < len(ps) and ps[j + 1] == ps[j] + 1:
            j += 1
        assert ps[i] % 2 == 0 and ps[j] % 2 == 0
        res.append([coords[ps[i] // 2], coords[ps[j] // 2]])
        i = j + 1
    return res


for (a, b), want in examples:
    assert oracle(a, b) == want == oracle_lattice(a, b)


def valid_lists(first, second):
    for lst in (first, second):
        for k, (s, e) in enumerate(lst):
            assert 0 <= s < e, lst
            if k:
                assert lst[k - 1][1] < s, lst


def check(first, second, want=None, names=None, use_oracle=True):
    valid_lists(first, second)
    if want is None and use_oracle:
        want = oracle(first, second)
    for name in (names or IMPLS):
        got = IMPLS[name]([x[:] for x in first], [x[:] for x in second])
        assert got == want, (name, first, second, got, want)


for (a, b), want in examples:
    check(a, b, want)
print("ví dụ đề: oracle tập điểm và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle tập điểm và 11 cài đặt đúng


### B. Edge case

- Một danh sách rỗng, hoặc cả hai rỗng không xảy ra (tổng độ dài `≥ 1`) nhưng ta vẫn thử.
- Một đoạn bao trọn nhiều đoạn của danh sách kia (`[0,10]` với `[1,2],[3,4]`).
- Hai đoạn chạm nhau ở một điểm (`[1,3]` và `[3,5]` ⇒ `[3,3]`).
- Hai đoạn không chạm (`[1,2]` và `[3,4]`) ⇒ `[]`.
- Hai danh sách giống hệt nhau ⇒ chính nó.
- Một đoạn giao nhiều đoạn của danh sách kia; điểm cuối bằng nhau.
- Toạ độ cực đoan `0` và `10^9`.
- `[0,1]` và `[1,2]`, `[2,3]` xen kẽ liên tiếp chạm điểm.

In [19]:
edge = [
    ([[1, 3], [5, 9]], []), ([], [[1, 3], [5, 9]]), ([], []), ([[0, 10]], [[1, 2], [3, 4]]), ([[1, 2], [3, 4]], [[0, 10]]),
    ([[1, 3]], [[3, 5]]), ([[3, 5]], [[1, 3]]), ([[1, 2]], [[3, 4]]), ([[1, 2], [5, 6]], [[1, 2], [5, 6]]), ([[1, 5]], [[1, 5]]),
    ([[0, 10 ** 9]], [[0, 10 ** 9]]), ([[0, 1]], [[1, 2]]), ([[0, 1], [2, 3], [4, 5]], [[1, 2], [3, 4], [5, 6]]), ([[1, 4], [6, 9]], [[2, 6], [7, 10]]),
    ([[0, 5], [7, 8], [10, 12]], [[5, 7], [8, 10], [12, 14]]), ([[2, 9]], [[1, 3], [4, 5], [6, 7], [8, 10]]), ([[1, 10 ** 9 - 1]], [[10 ** 9 - 1, 10 ** 9]]),
]
for a, b in edge:
    check(a, b)
print(len(edge), "edge case (kể cả toạ độ 10^9): mọi cài đặt khớp oracle tập điểm có nén toạ độ")

17 edge case (kể cả toạ độ 10^9): mọi cài đặt khớp oracle tập điểm có nén toạ độ


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [20]:
def bug_strict(first, second):                        # lo < hi: mất giao điểm
    i = j = 0
    res = []
    while i < len(first) and j < len(second):
        lo, hi = max(first[i][0], second[j][0]), min(first[i][1], second[j][1])
        if lo < hi:
            res.append([lo, hi])
        if first[i][1] < second[j][1]:
            i += 1
        else:
            j += 1
    return res


def bug_advance_by_start(first, second):              # tiến theo điểm ĐẦU nhỏ hơn
    i = j = 0
    res = []
    while i < len(first) and j < len(second):
        lo, hi = max(first[i][0], second[j][0]), min(first[i][1], second[j][1])
        if lo <= hi:
            res.append([lo, hi])
        if first[i][0] < second[j][0]:
            i += 1
        else:
            j += 1
    return res


def bug_advance_both(first, second):                  # tiến cả hai con trỏ
    i = j = 0
    res = []
    while i < len(first) and j < len(second):
        lo, hi = max(first[i][0], second[j][0]), min(first[i][1], second[j][1])
        if lo <= hi:
            res.append([lo, hi])
        i += 1
        j += 1
    return res


def bug_no_check(first, second):                      # không kiểm tra lo <= hi
    i = j = 0
    res = []
    while i < len(first) and j < len(second):
        res.append([max(first[i][0], second[j][0]), min(first[i][1], second[j][1])])
        if first[i][1] < second[j][1]:
            i += 1
        else:
            j += 1
    return res


def bug_swapped_minmax(first, second):                # đảo min/max
    i = j = 0
    res = []
    while i < len(first) and j < len(second):
        lo, hi = min(first[i][0], second[j][0]), max(first[i][1], second[j][1])
        if lo <= hi:
            res.append([lo, hi])
        if first[i][1] < second[j][1]:
            i += 1
        else:
            j += 1
    return res


def bug_sweep_end_first(first, second):               # sự kiện kết thúc đứng trước bắt đầu tại cùng toạ độ
    events = []
    for s, e in first + second:
        events.append((s, 1))
        events.append((e, 0))
    events.sort()
    depth, open_at, res = 0, None, []
    for x, kind in events:
        if kind == 1:
            depth += 1
            if depth == 2:
                open_at = x
        else:
            if depth == 2:
                res.append([open_at, x])
            depth -= 1
    return res


def bug_advance_larger(first, second):                # tiến đoạn kết thúc MUỘN hơn
    i = j = 0
    res = []
    while i < len(first) and j < len(second):
        lo, hi = max(first[i][0], second[j][0]), min(first[i][1], second[j][1])
        if lo <= hi:
            res.append([lo, hi])
        if first[i][1] > second[j][1]:
            i += 1
        else:
            j += 1
    return res


BUGS = [bug_strict, bug_advance_by_start, bug_advance_both, bug_no_check, bug_swapped_minmax, bug_sweep_end_first, bug_advance_larger]
probe = [([[1, 3]], [[3, 5]]), ([[0, 10]], [[1, 2], [3, 4]]), ([[1, 2]], [[3, 4]]), ([[1, 4], [6, 9]], [[2, 6], [7, 10]]),
         ([[1, 2], [4, 5]], [[0, 10]]), ([[0, 2], [5, 10]], [[1, 5], [8, 12]]), ex1]
for bug in BUGS:
    found = None
    for a, b in probe:
        try:
            got = bug(a, b)
        except IndexError as e:
            got = "IndexError"
        if got != oracle(a, b):
            found = (a, b, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:22s} sai trên {found[0]} / {found[1]}: ra {found[2]}, đúng {oracle(found[0], found[1])}")

bug_strict             sai trên [[1, 3]] / [[3, 5]]: ra [], đúng [[3, 3]]
bug_advance_by_start   sai trên [[0, 10]] / [[1, 2], [3, 4]]: ra [[1, 2]], đúng [[1, 2], [3, 4]]
bug_advance_both       sai trên [[0, 10]] / [[1, 2], [3, 4]]: ra [[1, 2]], đúng [[1, 2], [3, 4]]
bug_no_check           sai trên [[1, 2]] / [[3, 4]]: ra [[3, 2]], đúng []
bug_swapped_minmax     sai trên [[1, 3]] / [[3, 5]]: ra [[1, 5]], đúng [[3, 3]]
bug_sweep_end_first    sai trên [[1, 3]] / [[3, 5]]: ra [], đúng [[3, 3]]
bug_advance_larger     sai trên [[0, 10]] / [[1, 2], [3, 4]]: ra [[1, 2]], đúng [[1, 2], [3, 4]]


### D. Vét cạn mọi cặp danh sách nhỏ (oracle tập điểm)

Mọi danh sách hợp lệ trên toạ độ `{0, …, 8}`: chọn `2k` giá trị phân biệt tăng dần làm `s₁ < e₁ < s₂ < e₂ < …` (`2⁸ = 256` danh sách, kể cả rỗng). Mọi cặp `256² = 65 536` (kể cả cặp có danh sách rỗng). Oracle: tập điểm trên lưới nửa số nguyên.

In [21]:
def all_lists(max_coord):
    out = []
    values = list(range(max_coord + 1))
    for k in range(0, (max_coord + 1) // 2 + 1):
        for chosen in itertools.combinations(values, 2 * k):
            out.append([[chosen[2 * i], chosen[2 * i + 1]] for i in range(k)])
    return out


lists = all_lists(8)
print("số danh sách hợp lệ trên {0..8}:", len(lists))
assert len(lists) == 256
t0 = time.perf_counter()
cnt = nonempty = 0
main_impls = ["two_pointers", "sweep", "bisect", "merge_by_start", "brute", "Solution"]
for a in lists:
    pa = to_points(a)
    for b in lists:
        want = runs_to_intervals(pa & to_points(b))
        for name in main_impls:
            assert IMPLS[name](a, b) == want, (name, a, b, IMPLS[name](a, b), want)
        cnt += 1
        nonempty += bool(want)
print(cnt, "cặp:", len(main_impls), "cài đặt khớp oracle tập điểm; cặp có giao:", nonempty, ";", round(time.perf_counter() - t0, 1), "s")
assert cnt == 65536

số danh sách hợp lệ trên {0..8}: 256


65536 cặp: 6 cài đặt khớp oracle tập điểm; cặp có giao: 64143 ; 0.8 s


In [22]:
# các class nộp còn lại trên mẫu cặp cố định
rng = random.Random(98601)
others = ["SolutionTwoPointers", "SolutionSweep", "SolutionBisect", "SolutionMergeByStart", "SolutionBrute"]
for _ in range(5000):
    a, b = rng.choice(lists), rng.choice(lists)
    want = oracle(a, b)
    for name in others:
        assert IMPLS[name](a, b) == want, (name, a, b)
print("5000 cặp mẫu: các class nộp còn lại đều đúng")

5000 cặp mẫu: các class nộp còn lại đều đúng


### E. Tính chất / ngẫu nhiên (toạ độ tới `10^9`, seed cố định)

(1) Các cách `O(n+m)`/`O(n log m)`/sweep khớp brute `O(nm)`; (2) giao đối xứng: `A ∩ B = B ∩ A`; (3) giao với chính nó là chính nó; (4) kết quả sắp xếp, rời nhau (`end_i < start_{i+1}`), mỗi đoạn nằm trong một đoạn của mỗi danh sách; (5) số giao `≤ n + m − 1`; (6) tổng độ dài giao `≤ min(tổng độ dài A, tổng độ dài B)`; (7) dịch/co giãn toạ độ không đổi cấu trúc (khớp sau biến đổi).

In [23]:
def contained(seg, lst):
    return any(a <= seg[0] and seg[1] <= b for a, b in lst)


rng = random.Random(20240986)
t0 = time.perf_counter()
fast = ["two_pointers", "sweep", "bisect", "merge_by_start", "Solution"]
for it in range(300):
    n, m = rng.randint(0, 1000), rng.randint(0, 1000)
    if n + m == 0:
        n = 1
    hi = rng.choice([2 * (n + m) + 5, 10 ** 4, 10 ** 9])
    hi = max(hi, 2 * max(n, m) + 1)
    a, b = make_list(rng, n, hi), make_list(rng, m, hi)
    want = intersect_brute(a, b)                                                      # (1)
    for name in fast:
        assert IMPLS[name](a, b) == want, (name, it)
    assert Solution().intervalIntersection(b, a) == want                              # (2)
    assert Solution().intervalIntersection(a, a) == a                                 # (3)
    for k, seg in enumerate(want):                                                    # (4)
        assert seg[0] <= seg[1] and contained(seg, a) and contained(seg, b)
        if k:
            assert want[k - 1][1] < seg[0]
    assert len(want) <= max(0, n + m - 1)                                             # (5)
    la, lb, lw = (sum(e - s for s, e in x) for x in (a, b, want))
    assert lw <= min(la, lb)                                                          # (6)
    shifted = Solution().intervalIntersection([[3 * s + 7, 3 * e + 7] for s, e in a], [[3 * s + 7, 3 * e + 7] for s, e in b])
    assert shifted == [[3 * s + 7, 3 * e + 7] for s, e in want]                       # (7)
print("300 cặp danh sách ngẫu nhiên (n, m ≤ 1000, toạ độ tới 10^9): 7 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

300 cặp danh sách ngẫu nhiên (n, m ≤ 1000, toạ độ tới 10^9): 7 tính chất đúng; 12.9 s


In [24]:
# toạ độ nhỏ (nhiều điểm chạm): đối chiếu oracle tập điểm trên danh sách ngẫu nhiên dài hơn
rng = random.Random(98602)
t0 = time.perf_counter()
for _ in range(600):
    top = rng.choice([20, 40, 60])
    a = rng.choice(all_lists(8)) if rng.random() < 0.3 else make_list(rng, rng.randint(0, top // 2 - 1), top)
    b = make_list(rng, rng.randint(0, top // 2 - 1), top)
    want = oracle(a, b)
    for name in fast:
        assert IMPLS[name](a, b) == want, (name, a, b)
print("600 cặp toạ độ nhỏ, nhiều điểm chạm: khớp oracle tập điểm;", round(time.perf_counter() - t0, 1), "s")

600 cặp toạ độ nhỏ, nhiều điểm chạm: khớp oracle tập điểm; 0.1 s


### F. Giới hạn

`n = m = 1000`, toạ độ `0` và `10^9`; hai danh sách "đan xen" chạm nhau tại mọi điểm (`[3i, 3i+2]` và `[3i+2, 3i+3]` ⇒ `1999` giao điểm); một đoạn khổng lồ bao `1000` đoạn.

In [25]:
t0 = time.perf_counter()
n = 1000
# đan xen chạm điểm: first = [3i, 3i+2], second = [3i+2, 3i+3]: mỗi đoạn chạm đoạn kia ở cả hai đầu => 2n-1 giao điểm
first = [[3 * i, 3 * i + 2] for i in range(n)]
second = [[3 * i + 2, 3 * i + 3] for i in range(n)]
valid_lists(first, second)
want = sorted([[3 * i + 2, 3 * i + 2] for i in range(n)] + [[3 * i + 3, 3 * i + 3] for i in range(n - 1)])
assert len(want) == 2 * n - 1 and oracle(first, second) == want
for name in fast + ["brute"]:
    assert IMPLS[name](first, second) == want, name
huge = [[0, 10 ** 9]]
many = [[2 * i, 2 * i + 1] for i in range(1000)]
for name in fast + ["brute"]:
    assert IMPLS[name](huge, many) == many, name
    assert IMPLS[name](many, huge) == many, name
extreme = ([[0, 1], [10 ** 9 - 1, 10 ** 9]], [[1, 5], [10 ** 9 - 1, 10 ** 9]])
for name in fast:
    assert IMPLS[name](*extreme) == [[1, 1], [10 ** 9 - 1, 10 ** 9]], name
rng = random.Random(98603)
a, b = make_list(rng, 1000), make_list(rng, 1000)
base = intersect_brute(a, b)
for name in fast:
    assert IMPLS[name](a, b) == base, name
print("n = m = 1000: đan xen chạm điểm (1999 giao điểm), đoạn khổng lồ, toạ độ cực đoan, ngẫu nhiên: mọi cách đúng;", round(time.perf_counter() - t0, 2), "s")

n = m = 1000: đan xen chạm điểm (1999 giao điểm), đoạn khổng lồ, toạ độ cực đoan, ngẫu nhiên: mọi cách đúng; 0.18 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `lo < hi` (mất giao điểm)

In [26]:
a, b = ex1
print("bug:", bug_strict(a, b), "| đúng:", Solution().intervalIntersection(a, b))
assert [5, 5] not in bug_strict(a, b) and [5, 5] in Solution().intervalIntersection(a, b)

bug: [[1, 2], [8, 10], [15, 23]] | đúng: [[1, 2], [5, 5], [8, 10], [15, 23], [24, 24], [25, 25]]


**Vì sao sai:** đoạn **đóng** chứa cả điểm cuối; `[1,5] ∩ [5,10] = {5} = [5,5]`. **Đúng:** `lo <= hi`.

### Lỗi 2: tiến con trỏ theo điểm đầu

In [27]:
a, b = [[0, 10]], [[1, 2], [3, 4]]
print("bug:", bug_advance_by_start(a, b), "| đúng:", Solution().intervalIntersection(a, b))
assert bug_advance_by_start(a, b) != Solution().intervalIntersection(a, b)

bug: [[1, 2]] | đúng: [[1, 2], [3, 4]]


**Vì sao sai:** đoạn nào bị bỏ phải là đoạn **kết thúc sớm hơn** (nó không thể giao thêm gì). Bỏ theo điểm đầu có thể bỏ đoạn lớn `[0,10]` còn giao với các đoạn sau.

### Lỗi 3: tiến cả hai con trỏ

In [28]:
a, b = [[0, 10]], [[1, 2], [3, 4]]
print("bug:", bug_advance_both(a, b), "| đúng:", Solution().intervalIntersection(a, b))
assert bug_advance_both(a, b) == [[1, 2]]

bug: [[1, 2]] | đúng: [[1, 2], [3, 4]]


**Vì sao sai:** mỗi bước chỉ được bỏ **một** đoạn — đoạn còn lại có thể giao tiếp với đoạn kế tiếp của danh sách kia.

### Lỗi 4: không kiểm tra `lo <= hi`

In [29]:
a, b = [[1, 2]], [[3, 4]]
print("bug:", bug_no_check(a, b), "| đúng:", Solution().intervalIntersection(a, b))
assert bug_no_check(a, b) == [[3, 2]] and Solution().intervalIntersection(a, b) == []

bug: [[3, 2]] | đúng: []


**Vì sao sai:** nếu `lo > hi` thì hai đoạn không giao; ghi `[3, 2]` là đoạn vô nghĩa. **Đúng:** chỉ ghi khi `lo <= hi`.

### Lỗi 5: quét sự kiện với kết thúc trước bắt đầu

In [30]:
a, b = [[1, 5]], [[5, 9]]
print("bug:", bug_sweep_end_first(a, b), "| đúng:", Solution().intervalIntersection(a, b))
assert bug_sweep_end_first(a, b) == [] and Solution().intervalIntersection(a, b) == [[5, 5]]

bug: [] | đúng: [[5, 5]]


**Vì sao sai:** tại toạ độ `5` nếu xử lý "kết thúc" trước "bắt đầu", độ sâu hạ xuống `0` rồi mới lên `1`: không bao giờ đạt `2`. **Đúng:** sắp `(toạ độ, loại)` với bắt đầu (`0`) trước kết thúc (`1`).

### Lỗi 6 (bonus): đảo `min`/`max` hoặc tiến đoạn kết thúc muộn hơn

In [31]:
print("đảo min/max:", bug_swapped_minmax([[1, 4]], [[2, 6]]), "| tiến đoạn muộn:", bug_advance_larger(*ex1)[:3], "| đúng:", Solution().intervalIntersection([[1, 4]], [[2, 6]]))
assert bug_swapped_minmax([[1, 4]], [[2, 6]]) == [[1, 6]] and Solution().intervalIntersection([[1, 4]], [[2, 6]]) == [[2, 4]]
assert bug_advance_larger(*ex1) != Solution().intervalIntersection(*ex1)

đảo min/max: [[1, 6]] | tiến đoạn muộn: [[1, 2]] | đúng: [[2, 4]]


**Vì sao sai:** giao thu hẹp (`max` đầu, `min` cuối); còn hợp mới mở rộng (`min` đầu, `max` cuối).

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** hai (hay `k`) danh sách đoạn đã sắp xếp, rời nhau → **hai con trỏ** kiểu merge; nếu các đoạn không sắp xếp/không rời nhau → sắp xếp hoặc **quét sự kiện**.

**Mẹo:**

1. Giao `[max(đầu), min(cuối)]`; hợp lệ khi `lo <= hi`.
2. Bỏ đoạn kết thúc **sớm hơn**.
3. Đoạn đóng ⇒ `<=` (giao điểm).
4. Quét sự kiện: bắt đầu trước kết thúc tại cùng toạ độ (đoạn đóng).
5. Oracle độc lập: tập điểm trên lưới nửa số nguyên (hoặc nhân đôi toạ độ).
6. Đầu ra đã sắp và rời nhau — không cần gộp.

**Họ bài cùng pattern:** Merge Intervals (56), Insert Interval (57), Non-overlapping Intervals (435), Meeting Rooms II (253, Premium), Employee Free Time (759, Premium), Merge Sorted Array (88).

## 15. Biến thể

1. **Hợp** hai danh sách đoạn đóng (gộp các đoạn chạm/chồng nhau).
2. **Giao của `k` danh sách** (lặp lại giao hai danh sách).
3. **Tổng độ dài phần giao.**
4. **Đoạn mở** (`(a, b)`): giao rỗng khi chạm điểm — dùng `lo < hi`.

Cài đặt và đối chiếu oracle tập điểm:

In [32]:
def union_two(first, second):
    merged = sorted(first + second)
    res = []
    for s, e in merged:
        if res and s <= res[-1][1]:                      # chạm hoặc chồng (đoạn đóng): gộp
            res[-1][1] = max(res[-1][1], e)
        else:
            res.append([s, e])
    return res


def intersect_k(lists_):
    cur = lists_[0]
    for nxt in lists_[1:]:
        cur = intersect_two_pointers(cur, nxt)
    return cur


def overlap_length(first, second):
    return sum(e - s for s, e in intersect_two_pointers(first, second))


def intersect_open(first, second):
    """Biến thể 4: đoạn MỞ (a, b): giao khác rỗng khi lo < hi."""
    i = j = 0
    res = []
    while i < len(first) and j < len(second):
        lo, hi = max(first[i][0], second[j][0]), min(first[i][1], second[j][1])
        if lo < hi:
            res.append([lo, hi])
        if first[i][1] < second[j][1]:
            i += 1
        else:
            j += 1
    return res


rng = random.Random(15)
for _ in range(500):
    a = make_list(rng, rng.randint(0, 6), 30)
    b = make_list(rng, rng.randint(0, 6), 30)
    c = make_list(rng, rng.randint(0, 6), 30)
    assert union_two(a, b) == runs_to_intervals(to_points(a) | to_points(b))
    assert intersect_k([a, b, c]) == runs_to_intervals(to_points(a) & to_points(b) & to_points(c)) if a and b and c else True
    assert overlap_length(a, b) == sum(hi - lo for lo, hi in oracle(a, b))
    # đoạn mở: độ dài dương <=> khoảng mở giao nhau khác rỗng
    assert intersect_open(a, b) == [[lo, hi] for lo, hi in oracle(a, b) if lo < hi]
print("hợp, giao k danh sách, tổng độ dài giao, đoạn mở: khớp oracle tập điểm")

hợp, giao k danh sách, tổng độ dài giao, đoạn mở: khớp oracle tập điểm


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp | `max/min` từng cặp | `O(n·m)` | `O(k)` | `n, m` nhỏ |
| 2. Hai con trỏ | bỏ đoạn kết thúc sớm | `O(n+m)` | `O(1)` | **đáp án chính** |
| 3. Quét sự kiện | độ sâu = 2 | `O((n+m) log)` | `O(n+m)` | không cần sắp trước, `k` danh sách |
| 4. Chặt nhị phân | `bisect` trên `end` | `O(n log m + k)` | `O(m)` | `n ≪ m` |
| 5. Trộn theo điểm đầu | nhớ đoạn cuối kia | `O(n+m)` | `O(1)` | luồng dữ liệu |

Mọi cách khớp oracle tập điểm trên toàn bộ `65 536` cặp danh sách nhỏ và khớp nhau tới `n, m = 1000`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [33]:
print(Solution().intervalIntersection([[1, 3]], [[3, 5]]), Solution().intervalIntersection([[1, 2]], [[3, 4]]),
      Solution().intervalIntersection([[0, 10]], [[1, 2], [3, 4]]), Solution().intervalIntersection([], [[1, 2]]))

[[3, 3]] [] [[1, 2], [3, 4]] []


### Bài 2 (tính tay)

`first = [[1,4],[6,8]]`, `second = [[2,6],[7,9]]`. Chạy hai con trỏ trên giấy và liệt kê các giao. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hai con trỏ:

In [34]:
def two_pointers_todo(first, second):
    i = j = 0
    res = []
    while i < len(first) and j < len(second):
        lo = 0          # TODO 1: max của hai điểm đầu
        hi = 0          # TODO 2: min của hai điểm cuối
        if lo <= hi:
            pass        # TODO 3: ghi [lo, hi]
        pass            # TODO 4: tiến con trỏ của đoạn có điểm cuối nhỏ hơn
        break           # xoá dòng này khi làm xong
    return res


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(two_pointers_todo(a, b) == w for (a, b), w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **quét sự kiện** (bắt đầu trước kết thúc tại cùng toạ độ):

In [35]:
class MySolution:
    def intervalIntersection(self, firstList, secondList):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().intervalIntersection(a, b) == w for (a, b), w in examples)

### Bài 5 (mini challenge): hợp hai danh sách đoạn

Viết `union_ans(first, second)` trả về **hợp** của hai danh sách đoạn đóng (gộp các đoạn chạm hoặc chồng nhau, vd `[1,2]` và `[2,3]` → `[1,3]`). Kiểm bằng oracle tập điểm.

In [36]:
def union_ans(first, second):
    pass            # TODO


# Khi làm xong:
# assert union_ans([[1, 3], [6, 8]], [[2, 4], [8, 9]]) == [[1, 4], [6, 9]]

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: đoạn đóng — chạm một điểm vẫn là giao.
- Mức 2: `[1,3]` và `[3,5]` chung điểm `3`.
- Mức 3: `[[3, 3]] [] [[1, 2], [3, 4]] []`.

**Bài 2**

- Mức 1: `[1,4]` với `[2,6]` ⇒ `[2,4]`; `4 < 6` tiến `i`.
- Mức 2: `[6,8]` với `[2,6]` ⇒ `[6,6]`; `8 ≥ 6` tiến `j`; `[6,8]` với `[7,9]` ⇒ `[7,8]`.
- Mức 3: `[[2,4],[6,6],[7,8]]`.

**Bài 3**

- Mức 1: `lo = max(first[i][0], second[j][0])`.
- Mức 2: `hi = min(first[i][1], second[j][1])`; `res.append([lo, hi])`.
- Mức 3: `if first[i][1] < second[j][1]: i += 1 else: j += 1`.

**Bài 4**

- Mức 1: tạo sự kiện `(start, 0)` và `(end, 1)`.
- Mức 2: sắp xếp; `depth += 1` khi bắt đầu; mở giao khi `depth == 2`.
- Mức 3: khi kết thúc và `depth == 2` thì ghi `[open_at, x]`; rồi `depth -= 1`.

**Bài 5**

- Mức 1: sắp xếp theo điểm đầu.
- Mức 2: gộp khi `s <= res[-1][1]` (đoạn đóng chạm cũng gộp).
- Mức 3: `res[-1][1] = max(res[-1][1], e)`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [37]:
vals = (Solution().intervalIntersection([[1, 3]], [[3, 5]]), Solution().intervalIntersection([[1, 2]], [[3, 4]]),
        Solution().intervalIntersection([[0, 10]], [[1, 2], [3, 4]]), Solution().intervalIntersection([], [[1, 2]]))
print(vals)
assert vals == ([[3, 3]], [], [[1, 2], [3, 4]], [])
print("Bài 1 OK")

([[3, 3]], [], [[1, 2], [3, 4]], [])
Bài 1 OK


### Bài 2

In [38]:
res = Solution().intervalIntersection([[1, 4], [6, 8]], [[2, 6], [7, 9]])
print(res)
assert res == [[2, 4], [6, 6], [7, 8]] == oracle([[1, 4], [6, 8]], [[2, 6], [7, 9]])
print("Bài 2 OK")

[[2, 4], [6, 6], [7, 8]]
Bài 2 OK


### Bài 3

In [39]:
def two_pointers_answer(first, second):
    i = j = 0
    res = []
    while i < len(first) and j < len(second):
        lo = max(first[i][0], second[j][0])
        hi = min(first[i][1], second[j][1])
        if lo <= hi:
            res.append([lo, hi])
        if first[i][1] < second[j][1]:
            i += 1
        else:
            j += 1
    return res


assert all(two_pointers_answer(a, b) == w for (a, b), w in examples)
rng = random.Random(3)
for _ in range(300):
    a, b = make_list(rng, rng.randint(0, 6), 40), make_list(rng, rng.randint(0, 6), 40)
    assert two_pointers_answer(a, b) == oracle(a, b)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [40]:
class MySolutionAnswer:
    def intervalIntersection(self, firstList, secondList):
        events = []
        for s, e in firstList + secondList:
            events.append((s, 0))
            events.append((e, 1))
        events.sort()
        depth, open_at, res = 0, 0, []
        for x, kind in events:
            if kind == 0:
                depth += 1
                if depth == 2:
                    open_at = x
            else:
                if depth == 2:
                    res.append([open_at, x])
                depth -= 1
        return res


assert all(MySolutionAnswer().intervalIntersection(a, b) == w for (a, b), w in examples)
rng = random.Random(4)
for _ in range(300):
    a, b = make_list(rng, rng.randint(0, 6), 40), make_list(rng, rng.randint(0, 6), 40)
    assert MySolutionAnswer().intervalIntersection(a, b) == oracle(a, b)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [41]:
def union_ans(first, second):
    res = []
    for s, e in sorted(first + second):
        if res and s <= res[-1][1]:
            res[-1][1] = max(res[-1][1], e)
        else:
            res.append([s, e])
    return res


assert union_ans([[1, 3], [6, 8]], [[2, 4], [8, 9]]) == [[1, 4], [6, 9]]
rng = random.Random(5)
for _ in range(300):
    a, b = make_list(rng, rng.randint(0, 6), 40), make_list(rng, rng.randint(0, 6), 40)
    assert union_ans(a, b) == runs_to_intervals(to_points(a) | to_points(b))
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (giao hai danh sách đoạn đã sắp, rời nhau → hai con trỏ):**

```python
class Solution:
    def intervalIntersection(self, firstList, secondList):
        i = j = 0
        res = []
        while i < len(firstList) and j < len(secondList):
            lo = max(firstList[i][0], secondList[j][0])
            hi = min(firstList[i][1], secondList[j][1])
            if lo <= hi:
                res.append([lo, hi])
            if firstList[i][1] < secondList[j][1]:
                i += 1
            else:
                j += 1
        return res
```

**Checklist trước khi submit:**

- [ ] Giao `[max(đầu), min(cuối)]`, ghi khi `lo <= hi` (không `<`)?
- [ ] Tiến con trỏ của đoạn có điểm cuối **nhỏ hơn**?
- [ ] Chỉ tiến **một** con trỏ mỗi bước?
- [ ] Xử lý danh sách rỗng (vòng `while` đã bảo vệ)?
- [ ] Đã thử `[1,3]/[3,5] → [[3,3]]`, không chạm, đoạn bao trọn nhiều đoạn, ví dụ 1?
- [ ] Đầu ra đã sắp xếp và rời nhau (không cần gộp)?
- [ ] Đúng tên `class Solution`, `intervalIntersection(self, firstList, secondList)`, không in debug.